# Notebook 02 — ABC Inventory Analysis
**Tujuan:** Klasifikasi produk ke kelas A/B/C berdasarkan kontribusi revenue (Prinsip Pareto 80/20).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

OUT     = Path('../output')
FIGURES = OUT / 'figures'

BLUE  = '#2563EB'
RED   = '#DC2626'
AMBER = '#F59E0B'
GRAY  = '#6B7280'
LIGHT = '#93C5FD'
sns.set_theme(style='whitegrid', font_scale=1.1)

df = pd.read_parquet(OUT / 'df_clean.parquet')

# Identifikasi kolom revenue dan quantity
rev_col = [c for c in df.columns if 'order_item_total' in c or ('sales' in c and 'per' not in c)][0]
qty_col = [c for c in df.columns if 'order_item_quantity' in c or 'quantity' in c][0]
print(f'Revenue col : {rev_col}')
print(f'Quantity col: {qty_col}')
print(f'Loaded: {len(df):,} baris')

## 1. Agregasi Revenue per Produk

In [ ]:
abc = (
    df.groupby('product_name')
    .agg(
        total_revenue = (rev_col, 'sum'),
        total_units   = (qty_col, 'sum'),
        avg_price     = (rev_col, 'mean'),
        num_orders    = ('product_name', 'count')
    )
    .sort_values('total_revenue', ascending=False)
    .reset_index()
)

abc['cumulative_revenue']    = abc['total_revenue'].cumsum()
abc['cumulative_pct']        = abc['cumulative_revenue'] / abc['total_revenue'].sum() * 100
abc['revenue_pct']           = abc['total_revenue'] / abc['total_revenue'].sum() * 100

print(f'Produk unik: {len(abc):,}')
print(f'Total revenue: ${abc["total_revenue"].sum():,.0f}')

## 2. Klasifikasi A / B / C

In [ ]:
def assign_abc(cum_pct):
    if cum_pct <= 80:  return 'A'
    elif cum_pct <= 95: return 'B'
    else:               return 'C'

abc['abc_class'] = abc['cumulative_pct'].apply(assign_abc)

summary = abc.groupby('abc_class').agg(
    n_products    = ('product_name', 'count'),
    total_revenue = ('total_revenue', 'sum'),
    pct_products  = ('product_name', lambda x: len(x)/len(abc)*100),
    pct_revenue   = ('total_revenue', lambda x: x.sum()/abc['total_revenue'].sum()*100)
).round(2)

print('=== ABC CLASSIFICATION SUMMARY ===')
print(summary)

## 3. Pareto Chart

In [ ]:
# Ambil top 50 produk untuk keterbacaan
top_n = min(50, len(abc))
abc_plot = abc.head(top_n).copy()
class_colors = {'A': BLUE, 'B': AMBER, 'C': GRAY}
bar_colors = [class_colors[c] for c in abc_plot['abc_class']]

fig, ax1 = plt.subplots(figsize=(18, 6))
ax2 = ax1.twinx()

ax1.bar(range(top_n), abc_plot['revenue_pct'], color=bar_colors, alpha=0.85)
ax2.plot(range(top_n), abc_plot['cumulative_pct'], color=RED, linewidth=2.5,
         marker='o', markersize=3, label='Cumulative %')

ax2.axhline(80, color=RED,  linestyle='--', linewidth=1.5, alpha=0.7, label='80% threshold (A)')
ax2.axhline(95, color=AMBER, linestyle='--', linewidth=1.5, alpha=0.7, label='95% threshold (B)')

ax1.set_xlabel('Produk (diurutkan berdasarkan revenue)', fontsize=10)
ax1.set_ylabel('Revenue Contribution (%)', color=BLUE)
ax2.set_ylabel('Cumulative Revenue (%)', color=RED)
ax1.set_xticks([])
ax2.set_ylim(0, 110)
ax2.yaxis.set_major_formatter(mticker.PercentFormatter())

# Patch legend per kelas
import matplotlib.patches as mpatches
patches = [mpatches.Patch(color=v, label=f'Kelas {k}') for k, v in class_colors.items()]
ax1.legend(handles=patches, loc='upper right')
ax2.legend(loc='center right', fontsize=9)

ax1.set_title(f'ABC Analysis — Pareto Chart (Top {top_n} Produk)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES / 'A_abc_pareto.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: A_abc_pareto.png')

## 4. Distribusi Kelas A/B/C

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie: % produk per kelas
counts = abc['abc_class'].value_counts().sort_index()
axes[0].pie(counts.values,
            labels=[f'Kelas {c}' for c in counts.index],
            autopct='%1.1f%%',
            colors=[BLUE, AMBER, GRAY],
            wedgeprops={'width': 0.55}, startangle=90)
axes[0].set_title('Proporsi Produk per Kelas ABC', fontweight='bold')

# Bar: revenue per kelas
rev_by_class = abc.groupby('abc_class')['total_revenue'].sum().sort_index()
axes[1].bar(rev_by_class.index,
            rev_by_class.values / 1e6,
            color=[BLUE, AMBER, GRAY])
axes[1].set_ylabel('Total Revenue (juta USD)')
axes[1].set_title('Total Revenue per Kelas ABC', fontweight='bold')
for i, v in enumerate(rev_by_class.values / 1e6):
    axes[1].text(i, v + 0.05, f'${v:.1f}M', ha='center', fontweight='bold')

plt.suptitle('ABC Inventory Classification', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES / 'B_abc_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: B_abc_distribution.png')

## 5. Top 10 Produk Kelas A

In [ ]:
top10_a = abc[abc['abc_class']=='A'].head(10)[[
    'product_name','total_revenue','total_units','revenue_pct','cumulative_pct'
]].round(2)
print('=== TOP 10 PRODUK KELAS A ===')
print(top10_a.to_string(index=False))

## 6. Simpan df_abc

In [ ]:
abc.to_parquet(OUT / 'df_abc.parquet', index=False)
print(f'Tersimpan: df_abc.parquet — {len(abc):,} produk')
print(f'Validasi: cumulative_pct max = {abc["cumulative_pct"].max():.1f}%')